O pré-processamento típico segue essa ordem:

- Olhar os tipos (df.info()) e ver o que devia ser número/data mas veio como object/str

- Olhar valores únicos (df["COLUNA"].unique()) pra achar sujeira: espaços, formatos inconsistentes.

- Limpar (strip, trocar separador, etc.)

- Converter pro tipo certo

- Só então calcular estatísticas, agrupar, plotar

In [64]:
import pandas as pd

df = pd.read_csv("../dados/brutos/ufc_fight_stats.csv")

In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42012 entries, 0 to 42011
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EVENT       42012 non-null  str    
 1   BOUT        42012 non-null  str    
 2   ROUND       41970 non-null  str    
 3   FIGHTER     41970 non-null  str    
 4   KD          41970 non-null  float64
 5   SIG.STR.    41970 non-null  str    
 6   SIG.STR. %  41970 non-null  str    
 7   TOTAL STR.  41970 non-null  str    
 8   TD          41970 non-null  str    
 9   TD %        41970 non-null  str    
 10  SUB.ATT     41970 non-null  float64
 11  REV.        41970 non-null  float64
 12  CTRL        41970 non-null  str    
 13  HEAD        41970 non-null  str    
 14  BODY        41970 non-null  str    
 15  LEG         41970 non-null  str    
 16  DISTANCE    41970 non-null  str    
 17  CLINCH      41970 non-null  str    
 18  GROUND      41970 non-null  str    
dtypes: float64(3), str(16)
memory usage:

In [ ]:
resultados = pd.read_csv("../dados/processados/ufc_fight_results.csv")
chave = ["EVENT", "BOUT"]

# Garante que as chaves estão no mesmo formato nas duas tabelas
df["EVENT"] = df["EVENT"].str.strip()
df["BOUT"] = df["BOUT"].str.strip()

# Confere se a chave é única na tabela de referência (senão o merge duplica linhas)
print("Chave única em resultados?", not resultados.duplicated(chave).any())

antes = len(df)
df = df.merge(resultados[chave], on=chave, how="inner", validate="many_to_one")
print(f"Linhas: {antes} -> {len(df)}")


In [ ]:
# Conferência: todas as lutas da referência têm estatísticas?
lutas_stats = df[chave].drop_duplicates()
sem_stats = resultados[chave].merge(lutas_stats, on=chave, how="left", indicator=True)
print(sem_stats["_merge"].value_counts())   # o ideal é tudo em "both"


In [98]:
var = 'BODY'

df[var].unique()
# df[var].count()

<StringArray>
[  '3 of 5',   '6 of 6',   '3 of 4',   '0 of 0',   '2 of 3',   '7 of 9',
 '18 of 19',   '4 of 6',   '4 of 4',   '1 of 2',
 ...
 '19 of 31', '29 of 35',  '5 of 20', '24 of 51',  '4 of 19', '28 of 31',
 '20 of 20', '33 of 33', '35 of 42', '24 of 24']
Length: 379, dtype: str

In [90]:
df[var]

0        0:00
1        0:06
2        0:32
3        0:00
4        0:16
         ... 
41035    3:08
41036    0:18
41037    0:02
41038    2:25
41039    0:08
Name: CTRL, Length: 41040, dtype: str

Excluir os nulos e tranformar em Int


In [70]:
df = df.dropna(subset=["FIGHTER"])          # linhas sem lutador = linha vazia
df['KD'] = df['KD'].astype(int)


Uma coluna somente com o número de golpes significativos

In [71]:
df["SIG.STR."] = df["SIG.STR."].str.split(" of ").str[0].astype(int)

Trocando um texto de '10%' para 0.10

In [72]:
df["SIG.STR. %"] = pd.to_numeric(df["SIG.STR. %"].str.rstrip("%"), errors="coerce") / 100
df["TD %"]       = pd.to_numeric(df["TD %"].str.rstrip("%"), errors="coerce") / 100



Uma coluna somente com o número de golpes

In [73]:


partes = df["TOTAL STR."].str.split(" of ", expand=True).astype(int)

df["TOTAL STR. %"] = partes[0] / partes[1]   # 0 tentativas -> divisão por zero -> NaN
df["TOTAL STR."]   = partes[0]               # sobrescreve só no fim


Uma coluna somente com o número de Takedowns 

In [78]:
df["TD"] = df["TD"].str.split(" of ").str[0].astype(int)

Mudando tentativas de finalização e reversões para int

In [87]:
df = df.dropna(subset=["SUB.ATT"])        
df['SUB.ATT'] = df['SUB.ATT'].astype(int)

df = df.dropna(subset=["REV."])        
df['REV.'] = df['REV.'].astype(int)

Tempo de controle 

In [92]:
ctrl = df["CTRL"].replace("--", None)
partes = ctrl.str.split(":", expand=True).apply(pd.to_numeric)
df["CTRL_SEG"] = partes[0] * 60 + partes[1]


Total de golpes na cabeça

In [103]:
partes = df["HEAD"].str.split(" of ", expand=True).astype(int)

df["HEAD %"] = partes[0] / partes[1]  
df["HEAD"]   = partes[0] 

Total de golpes no corpo

In [104]:
partes = df["BODY"].str.split(" of ", expand=True).astype(int)

df["BODY %"] = partes[0] / partes[1]  
df["BODY"]   = partes[0] 

Total de golpes na perna

In [105]:
partes = df["LEG"].str.split(" of ", expand=True).astype(int)

df["LEG %"] = partes[0] / partes[1]  
df["LEG"]   = partes[0] 

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41040 entries, 0 to 41039
Data columns (total 32 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   EVENT          41040 non-null  str    
 1   BOUT           41040 non-null  str    
 2   ROUND          41040 non-null  str    
 3   FIGHTER        41040 non-null  str    
 4   KD             41040 non-null  int64  
 5   SIG.STR.       41040 non-null  int64  
 6   SIG.STR. %     40870 non-null  float64
 7   TOTAL STR.     41040 non-null  int64  
 8   TD             41040 non-null  int64  
 9   TD %           21906 non-null  float64
 10  SUB.ATT        41040 non-null  int64  
 11  REV.           41040 non-null  int64  
 12  CTRL           41040 non-null  str    
 13  HEAD           41040 non-null  int64  
 14  BODY           41040 non-null  int64  
 15  LEG            41040 non-null  int64  
 16  DISTANCE       41040 non-null  str    
 17  CLINCH         41040 non-null  str    
 18  GROUND         41

In [121]:
df.columns.tolist()


['EVENT',
 'BOUT',
 'ROUND',
 'FIGHTER',
 'KD',
 'SIG.STR.',
 'SIG.STR. %',
 'TOTAL STR.',
 'TD',
 'TD %',
 'SUB.ATT',
 'REV.',
 'CTRL',
 'HEAD',
 'BODY',
 'LEG',
 'DISTANCE',
 'CLINCH',
 'GROUND',
 'TOTAL STR. %',
 'CTRL_SEG',
 'HEAD %',
 'BODY %',
 'LEG %']